## How to access STAC Collections from the private STAC API at terrabyte - using TSM SSC as example

### Requirements:

1. You need access rights to the DSS container with the TSM/TDM data (`/dss/dsstbyfs03/pn56su/pn56su-dss-0028/`).
    - Load terrapi with `module use /dss/dsstbyfs01/pn56su/pn56su-dss-0020/usr/share/modules/files/; module load terrapi` and than list all available Container and request access for the TSX Container `terrapi restricted-data list-available` and `terrapi restricted-data request-access <NAME>`
2. Before you start querying the private STAC API with the [`terrapi-client`](https://github.com/DLR-terrabyte/terrapi-client) lib, you need to authentificate yourself with the [terrapi cli](https://docs.terrabyte.lrz.de/software/tools/terrapi/usage/terrapi/stac/auth/): ```terrapi stac auth```

-> `pip install git+https://github.com/DLR-terrabyte/terrapi-client.git@latest-release` into your python env.

-> `pip install git+https://github.com/DLR-terrabyte/terrapi.git@latest-release`

In [1]:
import pystac
import folium
import folium.plugins as folium_plugins
import geopandas as gpd
from odc.geo import geobox
from terrapi.client.stac import open_private_catalog

In [9]:
# Connect to the terrabyte private STAC API

# define private catalog
catalog = open_private_catalog()

# define search parameters like date, bounding box (bbox), instrument mode, and polarization
date = ["2022-01-01", "2022-12-30"]        # start and end data
bbox = [-85.66, 74.4, -79.8, 75.74]        # Lon,Lat of lower left and Lon,Lat of upper right corner

query={
    "sar:instrument_mode": {"eq": "SM"},         # ST, HS, SM, SL, or SC
    #"sar:polarizations": {"eq": ["HH", "VV"]},  # HH, HV, VV, or VH
}

# define STAC collection
results = catalog.search(
    collections=[f"pn56su-dss-0028.TSM_SSC"],         # pn56su-dss-0028.TSM_SSC or pn56su-dss-0028.TSM_MGD
    datetime=date,
    query=query,
    bbox=bbox
)

# search
items = results.item_collection_as_dict()
print("%s items found" % len(items['features']))

112 items found


## Inspect STAC Items

In [10]:
items = list(results.items())

In [4]:
items[0]

<Item id=TDX1_SAR__SSC______HS_S_SRA_20221230T231652_20221230T231653>

In [16]:
for i in items: 
    print(i.id)

TDX1_SAR__SSC______SM_S_SRA_20221230T221906_20221230T221914
TSX1_SAR__SSC______SM_D_SRA_20221230T125723_20221230T125733
TDX1_SAR__SSC______SM_S_SRA_20221229T223617_20221229T223625
TSX1_SAR__SSC______SM_D_SRA_20221220T124015_20221220T124025
TDX1_SAR__SSC______SM_S_SRA_20221219T221907_20221219T221915
TDX1_SAR__SSC______SM_D_SRA_20221219T125723_20221219T125733
TDX1_SAR__SSC______SM_S_SRA_20221218T223618_20221218T223626
TDX1_SAR__SSC______SM_D_SRA_20221209T124015_20221209T124025
TDX1_SAR__SSC______SM_S_SRA_20221208T221907_20221208T221915
TDX1_SAR__SSC______SM_D_SRA_20221208T125724_20221208T125734
TDX1_SAR__SSC______SM_S_SRA_20221207T223618_20221207T223626
TDX1_SAR__SSC______SM_D_SRA_20221128T124016_20221128T124026
TDX1_SAR__SSC______SM_S_SRA_20221127T221907_20221127T221915
TDX1_SAR__SSC______SM_D_SRA_20221127T125724_20221127T125734
TDX1_SAR__SSC______SM_S_SRA_20221126T223618_20221126T223626
TDX1_SAR__SSC______SM_D_SRA_20221117T124016_20221117T124026
TDX1_SAR__SSC______SM_S_SRA_20221116T221

In [11]:
map = folium.Map()
layer_control = folium.LayerControl(position='topright', collapsed=True)
fullscreen = folium_plugins.Fullscreen()
style = {'fillColor': '#00000000', "color": "#0000ff", "weight": 3}

footprints = folium.GeoJson(
    gpd.GeoDataFrame.from_features([item.to_dict() for item in items]).to_json(),
    name='Stac Item footprints',
    style_function=lambda x: style,
    control=True
)

footprints.add_to(map)
layer_control.add_to(map)
fullscreen.add_to(map)
map.fit_bounds(map.get_bounds())
map